# 행정공문 계층형 청킹 실습 노트북

이 노트북은 **생성형 AI 활용 시책제안 및 업무활용 우수사례 공모전 추진계획** 마크다운 문서를 대상으로 계층형 청킹을 실습하기 위한 파일입니다.

기존 법령 문서 실습이 `문서 제목 → 장 → 절 → 조` 구조였다면, 이 문서는 행정공문이므로 다음 구조를 사용합니다.

`문서 제목 → 본문 구역/붙임 → 항목(☐, 번호, 신청서명) → 세부항목(○)·표`

## 실습 목표

최종적으로 `chunks_1.json` 파일을 만듭니다.

청크 하나는 대략 다음 구조를 가집니다.

```json
{
  "chunk_id": "chunk_0001",
  "document_title": "...",
  "major_section": "공모개요",
  "item_title": "공모기간",
  "subitem_title": null,
  "level": 2,
  "chunk_type": "checkbox_item",
  "title": "공모기간",
  "content": "...",
  "source_path": "문서명 > 공모개요 > 공모기간",
  "start_line": 20,
  "end_line": 21
}
```

이 구조를 쓰면 나중에 RAG에서 “공모기간”, “심사기준”, “붙임 신청서”, “평가표” 같은 질문에 더 안정적으로 답할 수 있습니다.

## 1. 파일 경로 설정

같은 폴더에 마크다운 파일을 넣고 실행하세요.

결과 저장 파일명은 요청하신 대로 `chunks_1.json`입니다.

In [ ]:

from pathlib import Path
import re
import json
from datetime import datetime

# 같은 폴더에 마크다운 파일이 있을 때 사용할 파일명 후보입니다.
# 사용자가 파일명을 바꿨다면 candidates 안에 파일명을 추가하거나 INPUT_PATH를 직접 수정하면 됩니다.
candidates = [
    Path("생성형 AI 활용 시책제안 및 우수사례 공모전 추진계획(통보용).md"),
    Path("생성형 AI 활용 시책제안 및 우수사례 공모전 추진계획(통보용)(1).md"),
    Path("/mnt/data/생성형 AI 활용 시책제안 및 우수사례 공모전 추진계획(통보용)(1).md"),
]

INPUT_PATH = next((p for p in candidates if p.exists()), candidates[0])
OUTPUT_PATH = Path("chunks_1.json")

print("입력 파일:", INPUT_PATH)
print("파일 존재 여부:", INPUT_PATH.exists())
print("결과 파일:", OUTPUT_PATH)


## 2. 문서 일부 확인

먼저 실제 마크다운 문서가 어떤 줄 구조를 갖는지 확인합니다.

In [ ]:
text = INPUT_PATH.read_text(encoding="utf-8")
lines = text.splitlines()

for i, line in enumerate(lines[:60], start=1):
    print(f"{i:03d}: {line}")

## 3. 구조 후보 찾기

이 문서는 법령처럼 `제n조`가 아니라, 다음 기호들이 구조를 만듭니다.

- `###` : 본문 대구역
- `☐` : 주요 항목
- `○` : 세부 항목
- `붙임1`, `붙임2` : 첨부 서식
- `<table>` 또는 Markdown 표 : 별도 표 청크

In [ ]:
for i, line in enumerate(lines, start=1):
    stripped = line.strip()
    if stripped.startswith("###"):
        print(f"{i:03d}: SECTION  | {stripped}")
    elif stripped.startswith("☐"):
        print(f"{i:03d}: ITEM     | {stripped}")
    elif "○" in stripped:
        print(f"{i:03d}: SUBITEM  | {stripped}")
    elif stripped.startswith("붙임"):
        print(f"{i:03d}: ATTACH   | {stripped}")
    elif stripped.startswith("<table") or stripped.startswith("|"):
        print(f"{i:03d}: TABLE    | {stripped[:80]}")

## 4. 정규식과 보조 함수 만들기

아래 코드는 행정공문의 구조 기호를 인식하기 위한 준비 단계입니다.

초보자 관점에서 보면, 정규식은 “이 줄이 제목인지, 체크박스 항목인지, 표인지 알아보는 규칙”이라고 생각하면 됩니다.

In [ ]:

# 행정공문 구조 인식용 정규식입니다.
heading_pattern = re.compile(r"^#{1,6}\s+(.+)$")                 # # 제목, ### 추진배경
section_pattern = re.compile(r"^###\s+(.+)$")                    # ### 추진배경
attachment_pattern = re.compile(r"^(붙임\d+)\s*$")               # 붙임1, 붙임2
checkbox_pattern = re.compile(r"^☐\s*(.+)$")                    # ☐ 공모기간 : ...
bullet_pattern = re.compile(r"^(?:[-–]\s*)?○\s*(.+)$")          # - ○ (시책제안) ...
numbered_pattern = re.compile(r"^(\d+)\.\s*(.+)$")             # 1. 심사절차
md_table_start_pattern = re.compile(r"^\|.*\|\s*$")             # | 평가항목 | 배점 |
html_table_start_pattern = re.compile(r"^<table", re.I)          # <table>
html_table_end_pattern = re.compile(r"</table>\s*$", re.I)       # </table>
form_title_pattern = re.compile(r"^생성형 AI활용 .*신청서\((.+)\)\s*$")


def clean_text(text):
    """줄바꿈과 공백을 너무 과하게 없애지 않고, 앞뒤 공백만 정리합니다."""
    return text.strip()


def extract_title_from_item(raw_text):
    """☐ 또는 ○ 뒤의 문장에서 검색용 제목과 본문을 나눕니다."""
    text = raw_text.strip()

    # 예: (업무 패러다임 전환) chatGPT로 대표되는 ...
    m = re.match(r"^\(([^)]+)\)\s*(.*)$", text)
    if m:
        return m.group(1).strip(), m.group(2).strip()

    # 예: 공모기간 : 2025. 8. 18. ...
    if ":" in text:
        title, body = text.split(":", 1)
        return title.strip(), body.strip()

    return text.strip(), ""


def build_source_path(document_title, major_section=None, item_title=None, subitem_title=None):
    """문서 안에서 이 청크가 어디에 있는지 사람이 읽기 쉽게 표시합니다."""
    parts = [document_title, major_section, item_title, subitem_title]
    return " > ".join([p for p in parts if p])


def make_chunk(chunk_id, document_title, major_section, item_title, subitem_title, level, chunk_type, content, start_line, end_line):
    """청크 1개를 같은 형식의 딕셔너리로 만듭니다."""
    title_candidates = [subitem_title, item_title, major_section, document_title]
    title = next((x for x in title_candidates if x), "제목 없음")

    return {
        "chunk_id": f"chunk_{chunk_id:04d}",
        "document_title": document_title,
        "major_section": major_section,
        "item_title": item_title,
        "subitem_title": subitem_title,
        "level": level,
        "chunk_type": chunk_type,
        "title": title,
        "content": clean_text(content),
        "source_path": build_source_path(document_title, major_section, item_title, subitem_title),
        "start_line": start_line,
        "end_line": end_line,
    }


## 5. 정규식 간단 테스트

샘플 줄에 대해 어떤 구조로 인식되는지 확인합니다.

In [ ]:
sample_lines = [
    "### 공모개요",
    "☐ 공모기간 : 2025. 8. 18.(월) ~ 8. 29.(금) / 2주간",
    "- ○ (시책제안) 생성형 AI를 활용하여 군정 발전을 이끌 수 있는 정책 아이디어",
    "1. 심사절차",
    "붙임1",
    "<table>",
]

for s in sample_lines:
    print("\n원문:", s)
    for name, pattern in [
        ("section", section_pattern),
        ("attachment", attachment_pattern),
        ("checkbox", checkbox_pattern),
        ("bullet", bullet_pattern),
        ("numbered", numbered_pattern),
        ("html_table", html_table_start_pattern),
    ]:
        m = pattern.match(s)
        if m:
            print(name, "=>", m.groups())

## 6. 계층형 청킹 함수 구현

핵심 함수입니다.

한 줄씩 읽으면서 현재 위치를 기억합니다.

예를 들어 `### 공모개요`를 만나면 현재 대구역이 `공모개요`가 되고, 그 아래에서 `☐ 공모기간`을 만나면 `공모개요 > 공모기간` 청크가 만들어집니다.

In [ ]:

def hierarchical_chunk_admin_notice(md_text):
    """
    행정공문 마크다운을 계층형으로 청킹합니다.

    인식하는 구조:
    문서 제목 → 본문 구역/붙임 → 항목(☐, 번호, 신청서명) → 세부항목(○) 또는 표
    """
    lines = md_text.splitlines()

    # 1) 문서 제목 찾기: 처음 나오는 # 제목들을 합쳐서 사용합니다.
    heading_titles = []
    for line in lines[:20]:
        stripped = line.strip()
        m = heading_pattern.match(stripped)
        if m:
            title = m.group(1).strip()
            if title not in heading_titles:
                heading_titles.append(title)

    if heading_titles:
        document_title = " ".join(heading_titles[:2])
    else:
        document_title = "생성형 AI 활용 시책제안 및 우수사례 공모전 추진계획"

    chunks = []
    chunk_no = 1

    current_major = "문서 개요"
    current_item = None
    current_subitem = None
    current_level = 1
    current_type = "text"
    buffer = []
    start_line = None

    def flush(end_line):
        """지금까지 모은 buffer를 청크로 저장합니다."""
        nonlocal chunk_no, buffer, start_line, current_major, current_item, current_subitem, current_level, current_type

        content = "\n".join(buffer).strip()
        if not content:
            buffer = []
            start_line = None
            return

        chunks.append(
            make_chunk(
                chunk_id=chunk_no,
                document_title=document_title,
                major_section=current_major,
                item_title=current_item,
                subitem_title=current_subitem,
                level=current_level,
                chunk_type=current_type,
                content=content,
                start_line=start_line,
                end_line=end_line,
            )
        )
        chunk_no += 1
        buffer = []
        start_line = None

    i = 0
    while i < len(lines):
        line_no = i + 1
        line = lines[i]
        stripped = line.strip()

        # 빈 줄은 현재 청크 내부의 줄바꿈으로만 보존합니다.
        if not stripped:
            if buffer:
                buffer.append("")
            i += 1
            continue

        # 처음의 # 제목 줄은 문서 제목으로 이미 사용했으므로 본문 청크에서는 제외합니다.
        if heading_pattern.match(stripped) and not section_pattern.match(stripped):
            i += 1
            continue

        # 2) HTML 표 청크 처리
        if html_table_start_pattern.match(stripped):
            flush(line_no - 1)
            table_lines = [line]
            table_start = line_no
            while i + 1 < len(lines):
                i += 1
                table_lines.append(lines[i])
                if html_table_end_pattern.search(lines[i].strip()):
                    break
            chunks.append(
                make_chunk(
                    chunk_id=chunk_no,
                    document_title=document_title,
                    major_section=current_major,
                    item_title=current_item,
                    subitem_title="표",
                    level=4,
                    chunk_type="html_table",
                    content="\n".join(table_lines),
                    start_line=table_start,
                    end_line=i + 1,
                )
            )
            chunk_no += 1
            i += 1
            continue

        # 3) Markdown 표 청크 처리
        if md_table_start_pattern.match(stripped):
            flush(line_no - 1)
            table_lines = [line]
            table_start = line_no
            while i + 1 < len(lines) and md_table_start_pattern.match(lines[i + 1].strip()):
                i += 1
                table_lines.append(lines[i])
            chunks.append(
                make_chunk(
                    chunk_id=chunk_no,
                    document_title=document_title,
                    major_section=current_major,
                    item_title=current_item,
                    subitem_title="표",
                    level=4,
                    chunk_type="markdown_table",
                    content="\n".join(table_lines),
                    start_line=table_start,
                    end_line=i + 1,
                )
            )
            chunk_no += 1
            i += 1
            continue

        # 4) ### 추진배경 같은 본문 대구역
        m = section_pattern.match(stripped)
        if m:
            flush(line_no - 1)
            current_major = m.group(1).strip()
            current_item = None
            current_subitem = None
            current_level = 1
            current_type = "section"
            buffer = []
            start_line = None
            i += 1
            continue

        # 5) 붙임1, 붙임2
        m = attachment_pattern.match(stripped)
        if m:
            flush(line_no - 1)
            current_major = m.group(1).strip()
            current_item = None
            current_subitem = None
            current_level = 1
            current_type = "attachment"
            buffer = []
            start_line = None
            i += 1
            continue

        # 6) 신청서 제목
        m = form_title_pattern.match(stripped)
        if m:
            flush(line_no - 1)
            current_item = f"신청서({m.group(1).strip()})"
            current_subitem = None
            current_level = 2
            current_type = "form"
            buffer = [stripped]
            start_line = line_no
            i += 1
            continue

        # 7) 1. 심사절차, 2. 심사기준
        m = numbered_pattern.match(stripped)
        if m:
            flush(line_no - 1)
            current_item = m.group(2).strip()
            current_subitem = None
            current_level = 2
            current_type = "numbered_item"
            buffer = [stripped]
            start_line = line_no
            i += 1
            continue

        # 8) ☐ 항목
        m = checkbox_pattern.match(stripped)
        if m:
            flush(line_no - 1)
            item_title, rest = extract_title_from_item(m.group(1))
            current_item = item_title
            current_subitem = None
            current_level = 2
            current_type = "checkbox_item"
            buffer = [stripped]
            start_line = line_no
            i += 1
            continue

        # 9) ○ 세부항목
        m = bullet_pattern.match(stripped)
        if m:
            flush(line_no - 1)
            subitem_title, rest = extract_title_from_item(m.group(1))
            current_subitem = subitem_title
            current_level = 3
            current_type = "bullet_item"
            buffer = [stripped]
            start_line = line_no
            i += 1
            continue

        # 10) 그 밖의 줄은 현재 청크에 추가합니다.
        if start_line is None:
            start_line = line_no
        buffer.append(line)
        i += 1

    flush(len(lines))
    return chunks


## 7. 청킹 실행

문서를 실제로 청킹하고 앞부분을 확인합니다.

In [ ]:

text = INPUT_PATH.read_text(encoding="utf-8")
lines = text.splitlines()

chunks = hierarchical_chunk_admin_notice(text)

print("생성된 청크 수:", len(chunks))
print("\n앞의 5개 청크 미리보기")
for chunk in chunks[:5]:
    print("-" * 80)
    print("chunk_id:", chunk["chunk_id"])
    print("level:", chunk["level"])
    print("chunk_type:", chunk["chunk_type"])
    print("source_path:", chunk["source_path"])
    print("content preview:", chunk["content"][:200].replace("\n", " "))


## 8. 특정 구역 검증

`심사기준`, `공모기간`, `붙임1`처럼 원하는 단어가 포함된 청크를 찾아봅니다.

In [ ]:

# 원하는 구역만 골라서 확인하는 함수입니다.
def show_chunks_by_keyword(keyword):
    for chunk in chunks:
        haystack = json.dumps(chunk, ensure_ascii=False)
        if keyword in haystack:
            print("=" * 80)
            print("chunk_id:", chunk["chunk_id"])
            print("source_path:", chunk["source_path"])
            print("chunk_type:", chunk["chunk_type"])
            print(chunk["content"][:800])

show_chunks_by_keyword("심사기준")


## 9. JSON 저장

요청하신 파일명인 `chunks_1.json`으로 저장합니다.

In [ ]:

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print("저장 완료:", OUTPUT_PATH.resolve())
print("저장된 청크 수:", len(chunks))


## 10. Vector DB 적재용 구조 예시

실제 RAG에서는 전체 JSON을 그대로 임베딩하기보다, `pageContent`와 `metadata`를 나누어 저장하는 방식이 좋습니다.

In [ ]:

vector_docs = []

for chunk in chunks:
    page_content = f"""문서명: {chunk['document_title']}
출처: {chunk['source_path']}
청크유형: {chunk['chunk_type']}

{chunk['content']}
""".strip()

    metadata = {
        "chunk_id": chunk["chunk_id"],
        "document_title": chunk["document_title"],
        "major_section": chunk["major_section"],
        "item_title": chunk["item_title"],
        "subitem_title": chunk["subitem_title"],
        "level": chunk["level"],
        "chunk_type": chunk["chunk_type"],
        "source_path": chunk["source_path"],
        "start_line": chunk["start_line"],
        "end_line": chunk["end_line"],
    }

    vector_docs.append({
        "pageContent": page_content,
        "metadata": metadata,
    })

print("Vector DB 적재용 문서 수:", len(vector_docs))
print(json.dumps(vector_docs[0], ensure_ascii=False, indent=2)[:1000])
